In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd

from shapely.geometry import LineString, Point
from shapely.ops import split, substring

from dotenv import dotenv_values

basefolder = Path.cwd().parents[1]
srcfolder = basefolder / "src"

sys.path.insert(0, str(srcfolder))

from damo_afvoergebiedaanvoergebied.utils.general_functions import (
    split_waterways_by_endpoints,
)

In [ ]:
# Paths
config = dotenv_values("..\\..\\.env")
case_dir = Path(config["PATH_BASIS"], "brummen2")

dir_basisdata = case_dir / "0_basisdata"
dir_results = case_dir
dir_results.mkdir(parents=True, exist_ok=True)

In [ ]:
# # Paths

# case_dir = Path(
#     r"C:\Users\NL1C2G\Sweco AB\51024754_Vallei_en_Veluwe_Afwateringseenheden - General\Analyse\3_analyses\leuvenumse_beek"
# )

# base_dir = case_dir / "0_basisdata"
# result_dir = case_dir / "1_resultaat"
# result_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Read input data

duikers = gpd.read_file(
    dir_basisdata / "duiker.gpkg"
)

overige_watergangen = gpd.read_file(
    dir_basisdata / "overige_watergang.gpkg"
)

hydroobjecten = gpd.read_file(
    dir_basisdata / "hydroobject.gpkg"
)

In [ ]:
# Settings
MAX_DISTANCE = 3.0      # maximale afstand van duiker tot watergang
MIN_LINE_LENGTH = 0.2   # minimale lengte watergang in meters

# Only used to determine whether two points are
# geometrically identical.
POINT_TOLERANCE = 1e-9

In [ ]:
duikers_c = duikers[duikers["categorie"] == 3].copy()

In [ ]:
# Helper functions
def prepare_lines(gdf):
    """
    Keep only LineStrings and MultiLineStrings,
    explode multipart geometries, and remove very
    short geometries.
    """

    result = gdf.copy()

    result = result[
        result.geometry.notna()
        & ~result.geometry.is_empty
        & result.geometry.geom_type.isin(
            [
                "LineString",
                "MultiLineString",
            ]
        )
    ].copy()

    result = result.explode(
        index_parts=False,
        ignore_index=True,
    )

    result = result[
        result.geometry.notna()
        & ~result.geometry.is_empty
        & (
            result.geometry.geom_type
            == "LineString"
        )
        & (
            result.geometry.length
            > MIN_LINE_LENGTH
        )
    ].copy()

    return result.reset_index(drop=True)


def make_codes_unique(gdf):
    """
    Add a suffix to duplicate code values.
    """

    result = gdf.copy()

    if "code" not in result.columns:
        return result

    result["code"] = result["code"].astype(str)

    duplicate_codes = result["code"].duplicated(
        keep=False
    )

    occurrence = (
        result.groupby(
            "code",
            sort=False,
        )
        .cumcount()
    )

    result.loc[duplicate_codes, "code"] = (
        result.loc[duplicate_codes, "code"]
        + "-"
        + occurrence.loc[
            duplicate_codes
        ].astype(str)
    )

    return result

def add_extended_duikers(
    split_other_lines,
    extended_duiker_lines,
    working_crs,
):
    """
    Add extended duikers to overige_watergangen.
    """

    if extended_duiker_lines.empty:
        return split_other_lines

    split_other_lines = split_other_lines.copy()
    extended_duiker_lines = (
        extended_duiker_lines.copy()
    )

    # Add missing columns to both layers
    for column in extended_duiker_lines.columns:
        if column not in split_other_lines.columns:
            split_other_lines[column] = None

    for column in split_other_lines.columns:
        if column not in extended_duiker_lines.columns:
            extended_duiker_lines[column] = None

    extended_duiker_lines = (
        extended_duiker_lines[
            split_other_lines.columns
        ]
    )

    return gpd.GeoDataFrame(
        pd.concat(
            [
                split_other_lines,
                extended_duiker_lines,
            ],
            ignore_index=True,
        ),
        geometry="geometry",
        crs=working_crs,
    )

In [ ]:
def find_duiker_connections(
    duiker_lines,
    other_lines,
    hydro_lines,
    working_crs,
    max_distance,
):
    """
    Find the nearest target for every duiker endpoint.
    """

    # Create duiker endpoints
    endpoint_records = []

    for duiker_id, line in zip(
        duiker_lines.index,
        duiker_lines.geometry,
    ):
        endpoint_records.extend(
            [
                {
                    "duiker_id": duiker_id,
                    "endpoint_number": 0,
                    "geometry": Point(line.coords[0]),
                },
                {
                    "duiker_id": duiker_id,
                    "endpoint_number": 1,
                    "geometry": Point(line.coords[-1]),
                },
            ]
        )

    duiker_endpoints = gpd.GeoDataFrame(
        endpoint_records,
        geometry="geometry",
        crs=working_crs,
    )

    duiker_endpoints["endpoint_id"] = (
        duiker_endpoints.index
    )

    # Create overige-watergang endpoints
    endpoint_records = []

    for line_id, line in zip(
        other_lines.index,
        other_lines.geometry,
    ):
        endpoint_records.extend(
            [
                {
                    "target_line_id": line_id,
                    "target_side": "start",
                    "geometry": Point(line.coords[0]),
                },
                {
                    "target_line_id": line_id,
                    "target_side": "end",
                    "geometry": Point(line.coords[-1]),
                },
            ]
        )

    overige_endpoints = gpd.GeoDataFrame(
        endpoint_records,
        geometry="geometry",
        crs=working_crs,
    )

    # First search for overige-watergang endpoints
    endpoint_matches = gpd.sjoin_nearest(
        duiker_endpoints,
        overige_endpoints[
            [
                "target_line_id",
                "target_side",
                "geometry",
            ]
        ],
        how="left",
        max_distance=max_distance,
        distance_col="distance",
    )

    endpoint_matches = (
        endpoint_matches
        .sort_values(
            [
                "endpoint_id",
                "distance",
                "index_right",
            ],
            na_position="last",
        )
        .drop_duplicates(
            subset="endpoint_id",
            keep="first",
        )
        .copy()
    )

    endpoint_matches["target_layer"] = (
        "overige_watergangen"
    )

    endpoint_matches["match_type"] = (
        "endpoint_overige_watergang"
    )

    # Find endpoints without an endpoint match
    matched_ids = set(
        endpoint_matches.loc[
            endpoint_matches["distance"].notna(),
            "endpoint_id",
        ]
    )

    unmatched_endpoints = duiker_endpoints[
        ~duiker_endpoints["endpoint_id"].isin(
            matched_ids
        )
    ].copy()

    # Create combined network
    network_records = []

    for line_id, line in zip(
        other_lines.index,
        other_lines.geometry,
    ):
        network_records.append(
            {
                "target_layer": (
                    "overige_watergangen"
                ),
                "target_line_id": line_id,
                "geometry": line,
            }
        )

    for line_id, line in zip(
        hydro_lines.index,
        hydro_lines.geometry,
    ):
        network_records.append(
            {
                "target_layer": "hydroobjecten",
                "target_line_id": line_id,
                "geometry": line,
            }
        )

    network_lines = gpd.GeoDataFrame(
        network_records,
        geometry="geometry",
        crs=working_crs,
    ).reset_index(drop=True)

    network_lines["network_id"] = (
        network_lines.index
    )

    # Search the remaining endpoints on the network
    if not unmatched_endpoints.empty:

        network_matches = gpd.sjoin_nearest(
            unmatched_endpoints,
            network_lines[
                [
                    "network_id",
                    "target_layer",
                    "target_line_id",
                    "geometry",
                ]
            ],
            how="left",
            max_distance=max_distance,
            distance_col="distance",
        )

        network_matches = (
            network_matches
            .sort_values(
                [
                    "endpoint_id",
                    "distance",
                    "index_right",
                ],
                na_position="last",
            )
            .drop_duplicates(
                subset="endpoint_id",
                keep="first",
            )
            .copy()
        )

        network_matches["target_side"] = None
        network_matches["match_type"] = (
            "projectie_op_lijn"
        )

    else:

        network_matches = gpd.GeoDataFrame(
            columns=[
                "endpoint_id",
                "duiker_id",
                "endpoint_number",
                "target_layer",
                "target_line_id",
                "target_side",
                "distance",
                "geometry",
                "match_type",
            ],
            geometry="geometry",
            crs=working_crs,
        )

    endpoint_matches = endpoint_matches[
        [
            "endpoint_id",
            "duiker_id",
            "endpoint_number",
            "target_layer",
            "target_line_id",
            "target_side",
            "distance",
            "geometry",
            "match_type",
        ]
    ]

    network_matches = network_matches[
        [
            "endpoint_id",
            "duiker_id",
            "endpoint_number",
            "target_layer",
            "target_line_id",
            "target_side",
            "distance",
            "geometry",
            "match_type",
        ]
    ]

    all_matches = pd.concat(
        [
            endpoint_matches,
            network_matches,
        ],
        ignore_index=True,
    )

    all_matches["connected"] = (
        all_matches["distance"].notna()
    )

    return all_matches
    

In [ ]:
def keep_shortest_other_line_connection(
    all_matches,
    duiker_id_column="duiker_id",
    target_id_column="target_line_id",
    distance_column="distance",
):
    """
    If both endpoints of one duiker connect to the same
    overige_watergang, keep only the shortest connection.

    Other connections, such as connections to hydroobjecten,
    remain unchanged.
    """

    matches = all_matches.copy()

    required_columns = {
        duiker_id_column,
        target_id_column,
        distance_column,
        "target_layer",
        "connected",
    }

    missing_columns = (
        required_columns
        - set(matches.columns)
    )

    if missing_columns:
        raise ValueError(
            "Missing columns in all_matches: "
            f"{sorted(missing_columns)}"
        )

    other_matches = matches.loc[
        matches["connected"]
        & matches["target_layer"].eq(
            "overige_watergangen"
        )
    ].copy()

    if other_matches.empty:
        return matches

    # Preserve the original order as a deterministic
    # tie-breaker when distances are equal.
    other_matches["_original_order"] = range(
        len(other_matches)
    )

    other_matches = other_matches.sort_values(
        [
            duiker_id_column,
            target_id_column,
            distance_column,
            "_original_order",
        ],
        kind="stable",
    )

    # Keep the shortest connection per duiker and
    # overige_watergang.
    shortest_matches = (
        other_matches
        .drop_duplicates(
            subset=[
                duiker_id_column,
                target_id_column,
            ],
            keep="first",
        )
    )

    rejected_index = (
        other_matches.index.difference(
            shortest_matches.index
        )
    )

    # Do not remove rows. Mark the longer connection
    # as invalid so the original match structure remains.
    matches.loc[
        rejected_index,
        "connected",
    ] = False

    return matches

In [ ]:
def calculate_target_geometries(
    all_matches,
    other_lines,
    hydro_lines,
):
    """
    Calculate exact connection points and split points.
    """

    other_lookup = other_lines.geometry.to_dict()
    hydro_lookup = hydro_lines.geometry.to_dict()

    split_points = {
        "overige_watergangen": {},
        "hydroobjecten": {},
    }

    target_geometries = []

    for match in all_matches.itertuples(
        index=False
    ):

        if (
            not match.connected
            or pd.isna(match.distance)
        ):
            target_geometries.append(None)
            continue

        if (
            match.target_layer
            == "overige_watergangen"
        ):
            target_line = other_lookup[
                match.target_line_id
            ]

        else:
            target_line = hydro_lookup[
                match.target_line_id
            ]

        if (
            match.match_type
            == "endpoint_overige_watergang"
        ):

            if match.target_side == "start":
                target_point = Point(
                    target_line.coords[0]
                )
            else:
                target_point = Point(
                    target_line.coords[-1]
                )

        else:

            target_point = target_line.interpolate(
                target_line.project(
                    match.geometry
                )
            )

        target_geometries.append(target_point)

        line_points = split_points[
            match.target_layer
        ].setdefault(
            match.target_line_id,
            [],
        )

        if not any(
            existing.distance(target_point)
            < MIN_LINE_LENGTH
            for existing in line_points
        ):
            line_points.append(target_point)

    all_matches = all_matches.copy()

    all_matches["target_geometry"] = (
        target_geometries
    )

    return all_matches, split_points

In [ ]:
def create_extended_duikers(
    all_matches,
    original_duiker_lines,
    working_crs,
):
    """
    Create extended duikers only when at least one
    endpoint is connected.
    """

    extended_records = []

    for duiker_id, original_line in zip(
        original_duiker_lines.index,
        original_duiker_lines.geometry,
    ):

        matches = all_matches[
            all_matches["duiker_id"]
            == duiker_id
        ]

        connected_matches = matches[
            matches["connected"]
        ]

        # Do not add unconnected duikers
        if connected_matches.empty:
            continue

        new_coordinates = list(
            original_line.coords
        )

        for match in connected_matches.itertuples(
            index=False
        ):

            target_point = match.target_geometry

            if target_point is None:
                continue

            target_coordinate = tuple(
                target_point.coords
            )[0]

            if match.endpoint_number == 0:

                if (
                    target_point.distance(
                        Point(original_line.coords[0])
                    )
                    > POINT_TOLERANCE
                ):
                    new_coordinates.insert(
                        0,
                        target_coordinate,
                    )

            elif match.endpoint_number == 1:

                if (
                    target_point.distance(
                        Point(original_line.coords[-1])
                    )
                    > POINT_TOLERANCE
                ):
                    new_coordinates.append(
                        target_coordinate,
                    )

        # Remove consecutive duplicate points
        cleaned_coordinates = [
            new_coordinates[0]
        ]

        for coordinate in new_coordinates[1:]:

            if (
                Point(
                    cleaned_coordinates[-1]
                ).distance(
                    Point(coordinate)
                )
                > POINT_TOLERANCE
            ):
                cleaned_coordinates.append(
                    coordinate
                )

        geometry = LineString(
            cleaned_coordinates
        )

        attributes = (
            original_duiker_lines
            .loc[duiker_id]
            .drop(labels="geometry")
            .to_dict()
        )

        attributes["geometry"] = geometry
        attributes["bron_type"] = "duiker"
        attributes["bron_line_id"] = duiker_id

        extended_records.append(attributes)

    return gpd.GeoDataFrame(
        extended_records,
        geometry="geometry",
        crs=working_crs,
    )

In [ ]:
def split_network_lines(
    gdf,
    layer_name,
    split_points,
    min_line_length,
):
    """
    Split lines at points stored in split_points.

    Parameters
    ----------
    gdf : GeoDataFrame
        Original lines to split.

    layer_name : str
        Key used in split_points.

    split_points : dict
        Structure:
        {
            layer_name: {
                original_line_id: [Point, Point, ...]
            }
        }

    min_line_length : float
        Minimum length of output line parts.
    """

    output_records = []

    # Convert attributes once. This avoids iterrows() and
    # avoids repeatedly copying complete GeoDataFrame rows.
    attribute_columns = [
        column
        for column in gdf.columns
        if column != "geometry"
    ]

    attribute_records = (
        gdf[attribute_columns]
        .to_dict(orient="records")
    )

    line_points_by_id = split_points.get(
        layer_name,
        {},
    )

    for line_id, line, attributes in zip(
        gdf.index,
        gdf.geometry,
        attribute_records,
    ):

        if line is None or line.is_empty:
            continue

        if line.geom_type != "LineString":
            continue

        # Get the points belonging to this original line.
        points = line_points_by_id.get(
            line_id,
            [],
        )

        # Always include the original line endpoints
        # as split boundaries.
        split_distances = [
            0.0,
            float(line.length),
        ]

        for point in points:

            if point is None or point.is_empty:
                continue

            # Project the point onto this exact original line.
            projected_distance = float(
                line.project(point)
            )

            # Do not create very short pieces at either
            # end of the line.
            if (
                projected_distance
                <= min_line_length
            ):
                continue

            if (
                projected_distance
                >= line.length - min_line_length
            ):
                continue

            split_distances.append(
                projected_distance
            )

        # Remove nearly duplicate split distances.
        split_distances = sorted(
            {
                round(distance, 12)
                for distance in split_distances
            }
        )

        for part_number, (
            start_distance,
            end_distance,
        ) in enumerate(
            zip(
                split_distances[:-1],
                split_distances[1:],
            )
        ):

            if (
                end_distance - start_distance
                <= min_line_length
            ):
                continue

            geometry = substring(
                line,
                start_distance,
                end_distance,
                normalized=False,
            )

            if (
                geometry is None
                or geometry.is_empty
                or geometry.geom_type
                != "LineString"
                or geometry.length
                <= min_line_length
            ):
                continue

            output_record = attributes.copy()

            output_record["geometry"] = geometry
            output_record["is_connectie"] = False
            output_record["connectie_type"] = None
            output_record["bron_line_id"] = line_id
            output_record["deelnummer"] = part_number

            output_records.append(
                output_record
            )

    output_columns = [
        column
        for column in gdf.columns
        if column != "geometry"
    ]

    output_columns += [
        "is_connectie",
        "connectie_type",
        "bron_line_id",
        "deelnummer",
        "geometry",
    ]

    if not output_records:
        return gpd.GeoDataFrame(
            columns=output_columns,
            geometry="geometry",
            crs=gdf.crs,
        )

    return gpd.GeoDataFrame(
        output_records,
        geometry="geometry",
        crs=gdf.crs,
    ).reset_index(drop=True)

In [ ]:
def add_line_endpoints_to_hydro_split_points(
    source_lines,
    hydro_lines,
    split_points,
    max_distance,
    working_crs,
):
    """
    Add endpoints of source_lines as split points
    on nearby hydroobjecten.
    """

    endpoint_records = []

    for line in source_lines.geometry:

        if line is None or line.is_empty:
            continue

        endpoint_records.extend(
            [
                {
                    "geometry": Point(line.coords[0]),
                },
                {
                    "geometry": Point(line.coords[-1]),
                },
            ]
        )

    source_endpoints = gpd.GeoDataFrame(
        endpoint_records,
        geometry="geometry",
        crs=working_crs,
    )

    if source_endpoints.empty:
        return split_points

    hydro_for_join = hydro_lines[
        ["geometry"]
    ].copy()

    hydro_for_join["hydro_line_id"] = (
        hydro_for_join.index
    )

    matches = gpd.sjoin_nearest(
        source_endpoints,
        hydro_for_join,
        how="left",
        max_distance=max_distance,
        distance_col="distance",
    )

    for match in matches.itertuples(
        index=False
    ):

        if pd.isna(match.hydro_line_id):
            continue

        hydro_line_id = match.hydro_line_id

        hydro_line = hydro_lines.loc[
            hydro_line_id,
            "geometry",
        ]

        split_point = hydro_line.interpolate(
            hydro_line.project(match.geometry)
        )

        points_for_line = split_points[
            "hydroobjecten"
        ].setdefault(
            hydro_line_id,
            [],
        )

        if not any(
            existing.distance(split_point)
            < POINT_TOLERANCE
            for existing in points_for_line
        ):
            points_for_line.append(
                split_point
            )

    return split_points

In [ ]:
def add_extended_duikers(
    split_other_lines,
    extended_duiker_lines,
    working_crs,
):
    """
    Add extended duikers to overige_watergangen.
    """

    if extended_duiker_lines.empty:
        return split_other_lines

    split_other_lines = split_other_lines.copy()
    extended_duiker_lines = (
        extended_duiker_lines.copy()
    )

    # Add missing columns to both layers
    for column in extended_duiker_lines.columns:
        if column not in split_other_lines.columns:
            split_other_lines[column] = None

    for column in split_other_lines.columns:
        if column not in extended_duiker_lines.columns:
            extended_duiker_lines[column] = None

    extended_duiker_lines = (
        extended_duiker_lines[
            split_other_lines.columns
        ]
    )

    return gpd.GeoDataFrame(
        pd.concat(
            [
                split_other_lines,
                extended_duiker_lines,
            ],
            ignore_index=True,
        ),
        geometry="geometry",
        crs=working_crs,
    )

In [ ]:
def split_output_lines(
    other_lines,
    hydro_lines,
    split_points,
    extended_duiker_lines,
    working_crs,
):
    """
    Split overige watergangen, add extended duikers,
    then split hydroobjecten using the endpoints of
    the final overige_watergangen layer.
    """

    # Split original overige watergangen
    split_other_lines = split_network_lines(
        other_lines,
        "overige_watergangen",
        split_points,
        MIN_LINE_LENGTH,
    )

    split_other_lines["bron_type"] = "overige_watergang"

    # Add connected and extended duikers
    split_other_lines = add_extended_duikers(
        split_other_lines=split_other_lines,
        extended_duiker_lines=(
            extended_duiker_lines
        ),
        working_crs=working_crs,
    )

    # Create separate split points for hydroobjecten
    hydro_split_points = {
        "hydroobjecten": {}
    }

    hydro_split_points = (
        add_line_endpoints_to_hydro_split_points(
            source_lines=split_other_lines,
            hydro_lines=hydro_lines,
            split_points=hydro_split_points,
            max_distance=MAX_DISTANCE,
            working_crs=working_crs,
        )
    )

    # Split hydroobjecten only at endpoints of
    # the final overige_watergangen layer
    split_hydro_lines = split_network_lines(
        hydro_lines,
        "hydroobjecten",
        hydro_split_points,
        MIN_LINE_LENGTH,
    )

    # Required source type
    split_hydro_lines["bron_type"] = (
        "hydroobject"
    )

    return (
        split_hydro_lines,
        split_other_lines,
    )

In [ ]:
def duiker_to_linestring(
    hydroobject,
    overige_watergang,
    duiker,
):
    """
    Convert connected duikers to LineStrings and
    add them to overige_watergang.
    """

    working_crs = duiker.crs

    overige_watergang = (
        overige_watergang.to_crs(working_crs)
    )

    hydroobject = (
        hydroobject.to_crs(working_crs)
    )

    # Prepare lines
    other_lines = prepare_lines(
        overige_watergang
    )

    hydro_lines = prepare_lines(
        hydroobject
    )

    duiker_lines = prepare_lines(
        duiker
    )

    # Find duiker connections
    all_matches = find_duiker_connections(
        duiker_lines,
        other_lines,
        hydro_lines,
        working_crs,
        MAX_DISTANCE,
    )
    # Step 1b: prevent one duiker from connecting
    # to the same overige_watergang on both sides
    all_matches = (
        keep_shortest_other_line_connection(
            all_matches
        )
    )

    # Calculate target points
    all_matches, split_points = (
        calculate_target_geometries(
            all_matches,
            other_lines,
            hydro_lines,
        )
    )

    # Create connected and extended duikers
    extended_duiker_lines = (
        create_extended_duikers(
            all_matches,
            duiker_lines,
            working_crs,
        )
    )

    # Split output layers
    (
        split_hydro_lines,
        split_other_lines,
    ) = split_output_lines(
        other_lines=other_lines,
        hydro_lines=hydro_lines,
        split_points=split_points,
        extended_duiker_lines=(
            extended_duiker_lines
        ),
        working_crs=working_crs,
    )

    # Remove temporary columns
    added_columns = [
        "bronhouder",
        "beheerregister_waterschap",
        "beheerregister_derden",
        "code_waterschap",
        "is_connectie",
        "connectie_type",
        "is_duiker_extension",
        "bron_line_id",
        "deelnummer",
    ]

    split_other_lines = (
        split_other_lines.drop(
            columns=added_columns,
            errors="ignore",
        )
    )

    split_hydro_lines = (
        split_hydro_lines.drop(
            columns=added_columns,
            errors="ignore",
        )
    )

    # Make codes unique after splitting
    split_other_lines = make_codes_unique(
        split_other_lines
    )

    split_hydro_lines = make_codes_unique(
        split_hydro_lines
    )

    return (
        split_hydro_lines,
        split_other_lines,
        extended_duiker_lines,
    )

In [ ]:
hydroobject, overige_watergang, duiker = (
    duiker_to_linestring(
        hydroobject=hydroobjecten,
        overige_watergang=overige_watergangen,
        duiker=duikers_c,
    )
)

In [ ]:
# Output paths
output_duikers = (
    dir_results / "duiker_test.gpkg"
)

output_overige_watergangen = (
    dir_results / "overige_watergangen_test.gpkg"
)

output_hydroobjecten = (
    dir_results / "hydroobjecten_test.gpkg"
)

In [ ]:
hydroobject.to_file(
    output_hydroobjecten,
    driver="GPKG",
)
overige_watergang.to_file(
    output_overige_watergangen,
    driver="GPKG",
)